# Branding Document Generator

This notebook builds a **fine-tuned, rule-based branding document** for a startup.

It is designed to work from structured startup inputs, not from an LLM.
The notebook can optionally use Kaggle reference datasets for visual identity and brand examples, but the final document is generated from the startup brief, brand rules, and conditional logic.

## What this notebook produces
- Brand strategy: purpose, vision, mission, values
- Brand identity: logo, colors, typography, imagery
- Brand communication: voice, tone, tagline
- Brand application: examples for website, social, marketing materials
- Exportable outputs: Markdown, and optional DOCX/PDF if dependencies are installed


In [34]:
BRAND_PERSONALITY_OPTIONS = [
    "premium",
    "friendly",
    "bold",
    "innovative",
    "trustworthy",
    "minimal",
    "playful",
    "professional",
]

TONE_OPTIONS = ["formal", "conversational", "inspirational", "technical", "luxury", "approachable"]
USAGE_CHANNELS = ["website", "instagram", "linkedin", "facebook", "x", "packaging", "presentations", "ads"]
CORE_VALUES = ["innovation", "trust", "quality", "sustainability", "speed", "simplicity", "community", "excellence"]

if HAS_WIDGETS:
    def build_startup_brand_form() -> Dict[str, Any]:
        company_name = widgets.Text(description="Company", placeholder="Startup name")
        startup_brief = widgets.Textarea(
            description="Brief",
            placeholder="Describe what the startup does, who it is for, and any idea you already have.",
            layout=widgets.Layout(width="100%", height="120px"),
        )
        color_preferences = widgets.Text(description="Colors", placeholder="Optional preferred colors or HEX codes")
        tagline = widgets.Text(description="Message", placeholder="Optional message, slogan, or style hint")
        industry = widgets.Dropdown(
            description="Industry",
            options=["auto", "technology", "fashion", "food", "health", "education", "finance", "beauty", "other"],
            value="auto",
        )

        form = {
            "company_name": company_name,
            "startup_brief": startup_brief,
            "color_preferences": color_preferences,
            "tagline": tagline,
            "industry": industry,
        }

        if display is not None:
            display(widgets.VBox([
                widgets.HTML("<h3>Startup Brand Input Form</h3><p>Write a short brief. The notebook will generate the full brand strategy, identity, and layout for you.</p>"),
                company_name,
                startup_brief,
                color_preferences,
                tagline,
                industry,
            ]))
        return form
else:
    def build_startup_brand_form() -> Dict[str, Any]:
        return {}

startup_brand_form = build_startup_brand_form()
print("Startup input form initialized.")


Startup input form initialized.


In [2]:
BRAND_PERSONALITY_OPTIONS = [
    "premium",
    "friendly",
    "bold",
    "innovative",
    "trustworthy",
    "minimal",
    "playful",
    "professional",
]

TONE_OPTIONS = ["formal", "conversational", "inspirational", "technical", "luxury", "approachable"]
USAGE_CHANNELS = ["website", "instagram", "linkedin", "facebook", "x", "packaging", "presentations", "ads"]
CORE_VALUES = ["innovation", "trust", "quality", "sustainability", "speed", "simplicity", "community", "excellence"]

if HAS_WIDGETS:
    def build_startup_brand_form() -> Dict[str, Any]:
        company_name = widgets.Text(description="Company", placeholder="Startup name")
        industry = widgets.Dropdown(
            description="Industry",
            options=["technology", "fashion", "food", "health", "education", "finance", "beauty", "other"],
            value="technology",
        )
        mission = widgets.Textarea(description="Mission", placeholder="What does the company do?", layout=widgets.Layout(width="100%", height="80px"))
        vision = widgets.Textarea(description="Vision", placeholder="What future does the brand want to create?", layout=widgets.Layout(width="100%", height="80px"))
        target_audience = widgets.Textarea(description="Audience", placeholder="Who is the brand for?", layout=widgets.Layout(width="100%", height="80px"))
        competitors = widgets.Text(description="Competitors", placeholder="Competitor names separated by commas")
        tagline = widgets.Text(description="Tagline", placeholder="Optional slogan")
        color_preferences = widgets.Text(description="Colors", placeholder="Preferred colors or HEX codes")
        typography_preferences = widgets.Text(description="Typography", placeholder="Font style preferences")
        imagery_style = widgets.Dropdown(description="Imagery", options=["minimal", "editorial", "corporate", "playful", "luxury", "authentic"], value="minimal")
        personality = widgets.SelectMultiple(description="Personality", options=BRAND_PERSONALITY_OPTIONS, value=("professional", "trustworthy"))
        tone = widgets.Dropdown(description="Tone", options=TONE_OPTIONS, value="professional")
        values = widgets.SelectMultiple(description="Values", options=CORE_VALUES, value=("quality", "trust"))
        channels = widgets.SelectMultiple(description="Channels", options=USAGE_CHANNELS, value=("website", "instagram", "linkedin"))

        form = {
            "company_name": company_name,
            "industry": industry,
            "mission": mission,
            "vision": vision,
            "target_audience": target_audience,
            "competitors": competitors,
            "tagline": tagline,
            "color_preferences": color_preferences,
            "typography_preferences": typography_preferences,
            "imagery_style": imagery_style,
            "personality": personality,
            "tone": tone,
            "values": values,
            "channels": channels,
        }

        if display is not None:
            display(widgets.VBox([
                widgets.HTML("<h3>Startup Brand Input Form</h3>"),
                company_name, industry, mission, vision, target_audience, competitors,
                tagline, color_preferences, typography_preferences, imagery_style,
                personality, tone, values, channels,
            ]))
        return form
else:
    def build_startup_brand_form() -> Dict[str, Any]:
        return {}

startup_brand_form = build_startup_brand_form()
print("Startup input form initialized.")


Startup input form initialized.


In [39]:
def normalize_text(value: Any) -> str:
    return str(value or "").strip()


def normalize_csv_list(value: Any) -> List[str]:
    text = normalize_text(value)
    if not text:
        return []
    return [item.strip() for item in text.split(",") if item.strip()]


def infer_industry_from_prompt(prompt: str) -> str:
    prompt_l = prompt.lower()
    keyword_map = {
        "technology": ["ai", "software", "app", "platform", "automation", "data", "saas", "digital", "tech"],
        "health": ["health", "clinic", "medical", "care", "wellness", "fitness", "therapy"],
        "finance": ["finance", "fintech", "bank", "payment", "wallet", "investment", "accounting"],
        "education": ["education", "learn", "learning", "school", "course", "student", "training"],
        "fashion": ["fashion", "clothing", "wear", "style", "apparel", "brand"],
        "food": ["food", "restaurant", "meal", "snack", "drink", "cafe", "delivery"],
        "beauty": ["beauty", "skincare", "cosmetic", "salon", "hair", "makeup"],
    }
    for industry, keywords in keyword_map.items():
        if any(keyword in prompt_l for keyword in keywords):
            return industry
    return "other"


def infer_startup_profile_from_prompt(prompt: str, company_name: str = "", color_preferences: str = "", tagline: str = "") -> Dict[str, Any]:
    prompt = normalize_text(prompt)
    industry = infer_industry_from_prompt(prompt)
    rules = INDUSTRY_RULES.get(industry, INDUSTRY_RULES["other"])

    prompt_l = prompt.lower()
    if not company_name:
        if "startup named" in prompt_l:
            company_name = prompt_l.split("startup named", 1)[1].split()[0].strip(".,;:!?\"'()[]{}") or "Startup"
        elif "name is" in prompt_l:
            company_name = prompt_l.split("name is", 1)[1].split()[0].strip(".,;:!?\"'()[]{}") or "Startup"
        else:
            company_name = "Startup"

    audience = ""
    for cue in ["for ", "dedicated for ", "targeting ", "to help ", "for startups", "for businesses", "for small businesses"]:
        if cue in prompt_l:
            audience = prompt[prompt_l.index(cue) + len(cue):].split(".")[0].split(",")[0].strip()
            break
    if not audience:
        audience = "people and organizations that need the startup's core service"

    mission = prompt.rstrip(".") if prompt else f"Build a clear {industry} solution for the target audience."
    vision = f"Become a trusted {industry} brand known for {rules['voice']}."
    if not tagline:
        tagline = f"A clear promise for {company_name or 'the startup'} that reflects its value."

    if not color_preferences:
        color_preferences = ""

    return {
        "company_name": company_name,
        "industry": industry,
        "mission": mission,
        "vision": vision,
        "target_audience": audience,
        "competitors": [],
        "tagline": tagline,
        "color_preferences": color_preferences,
        "typography_preferences": "",
        "imagery_style": rules["imagery"],
        "brand_personality": ["professional", "trustworthy", "innovative"],
        "tone_of_voice": "professional",
        "core_values": ["quality", "trust", "innovation"],
        "usage_channels": ["website", "instagram", "linkedin"],
    }


def extract_form_data(form: Dict[str, Any]) -> Dict[str, Any]:
    if not form:
        return {}

    data = {
        "company_name": normalize_text(form["company_name"].value),
        "startup_brief": normalize_text(form["startup_brief"].value),
        "industry": normalize_text(form["industry"].value).lower(),
        "target_audience": normalize_text(form["target_audience"].value),
        "tagline": normalize_text(form["tagline"].value),
        "color_preferences": normalize_text(form["color_preferences"].value),
    }
    return data


def validate_brand_inputs(data: Dict[str, Any]) -> List[str]:
    errors: List[str] = []
    required_fields = ["startup_brief"]
    for field_name in required_fields:
        if not normalize_text(data.get(field_name)):
            errors.append(f"Missing required field: {field_name}")

    return errors


print("Parsing and validation helpers are ready.")


Parsing and validation helpers are ready.


In [40]:
def build_brand_document_structure() -> Dict[str, Any]:
    return {
        "Executive Summary": "",
        "Brand Strategy": {
            "purpose": "",
            "vision": "",
            "mission": "",
            "core_values": [],
        },
        "Brand Identity": {
            "logo_guidelines": "",
            "color_palette": [],
            "typography": [],
            "imagery_style": "",
        },
        "Brand Communication": {
            "voice": "",
            "tone": "",
            "tagline": "",
        },
        "Brand Application": {
            "website": "",
            "social_media": "",
            "marketing_materials": "",
            "presentation": "",
        },
        "Target Audience": "",
        "Brand Personality": [],
        "Visual Identity Guidelines": "",
        "Tone of Voice": "",
        "Brand Positioning": "",
        "Key Messaging": [],
        "Usage Guidelines": [],
    }


INDUSTRY_RULES = {
    "technology": {
        "voice": "smart, clear, innovative",
        "palette": ["#111827", "#2563EB", "#0EA5E9", "#F8FAFC"],
        "typography": ["Headings: modern sans-serif", "Body: clean sans-serif"],
        "imagery": "product-led, high-clarity, digital-first visuals",
        "positioning": "Position the brand as modern, reliable, and scalable.",
    },
    "fashion": {
        "voice": "stylish, aspirational, confident",
        "palette": ["#111111", "#D4AF37", "#F5F5F5", "#C08457"],
        "typography": ["Headings: elegant serif or high-contrast sans", "Body: refined sans-serif"],
        "imagery": "editorial, premium, highly curated visuals",
        "positioning": "Position the brand as distinctive, expressive, and visually memorable.",
    },
    "food": {
        "voice": "warm, sensory, inviting",
        "palette": ["#7C2D12", "#F97316", "#FDE68A", "#FFF7ED"],
        "typography": ["Headings: friendly display sans", "Body: readable rounded sans"],
        "imagery": "fresh, appetizing, lifestyle-driven visuals",
        "positioning": "Position the brand as tasty, trustworthy, and approachable.",
    },
    "health": {
        "voice": "reassuring, expert, calm",
        "palette": ["#0F766E", "#14B8A6", "#E0F2FE", "#F8FAFC"],
        "typography": ["Headings: clean sans-serif", "Body: highly readable sans-serif"],
        "imagery": "clean, bright, clinical, human-centered visuals",
        "positioning": "Position the brand as credible, caring, and safe.",
    },
    "education": {
        "voice": "helpful, encouraging, accessible",
        "palette": ["#1D4ED8", "#7C3AED", "#E0E7FF", "#F8FAFC"],
        "typography": ["Headings: approachable sans-serif", "Body: easy-to-read sans-serif"],
        "imagery": "learner-focused, optimistic, illustrative visuals",
        "positioning": "Position the brand as supportive, clear, and growth-oriented.",
    },
    "finance": {
        "voice": "trustworthy, precise, professional",
        "palette": ["#0F172A", "#1E3A8A", "#16A34A", "#F8FAFC"],
        "typography": ["Headings: authoritative sans-serif", "Body: structured sans-serif"],
        "imagery": "structured, confident, data-led visuals",
        "positioning": "Position the brand as dependable, intelligent, and secure.",
    },
    "beauty": {
        "voice": "sensory, elegant, confident",
        "palette": ["#111827", "#DB2777", "#FBCFE8", "#FFF1F2"],
        "typography": ["Headings: elegant display font", "Body: soft, modern sans-serif"],
        "imagery": "premium, close-up, polished lifestyle visuals",
        "positioning": "Position the brand as expressive, refined, and desirable.",
    },
    "other": {
        "voice": "clear, adaptable, professional",
        "palette": ["#111827", "#2563EB", "#10B981", "#F8FAFC"],
        "typography": ["Headings: clean sans-serif", "Body: readable sans-serif"],
        "imagery": "balanced, product-relevant, consistent visuals",
        "positioning": "Position the brand with clarity, relevance, and consistency.",
    },
}

COLOR_NAME_TO_HEX = {
    "beige": "#F5F5DC",
    "brown": "#8B5A2B",
    "sand": "#C2B280",
    "tan": "#D2B48C",
    "cream": "#FFFDD0",
    "black": "#111111",
    "white": "#FFFFFF",
    "navy": "#1D2D50",
    "blue": "#2563EB",
    "green": "#16A34A",
    "gray": "#6B7280",
    "grey": "#6B7280",
    "gold": "#D4AF37",
    "golden": "#D4AF37",
    "olive": "#808000",
    "burgundy": "#800020",
    "rose": "#E11D48",
}


def recommend_typography(industry: str, typography_preferences: str) -> List[str]:
    rules = INDUSTRY_RULES.get(industry, INDUSTRY_RULES["other"])
    recommendations = list(rules["typography"])
    if typography_preferences:
        recommendations.append(f"Preference note: {typography_preferences}")
    return recommendations


def recommend_color_palette(industry: str, color_preferences: str) -> List[str]:
    rules = INDUSTRY_RULES.get(industry, INDUSTRY_RULES["other"])
    palette: List[str] = []

    pref_text = normalize_text(color_preferences).lower()
    for color_name, hex_code in COLOR_NAME_TO_HEX.items():
        if color_name in pref_text and hex_code not in palette:
            palette.append(hex_code)

    for color in rules["palette"]:
        if color not in palette:
            palette.append(color)

    if color_preferences:
        palette.append(f"Preference note: {color_preferences}")
    return palette


def generate_brand_document(data: Dict[str, Any]) -> Dict[str, Any]:
    structure = build_brand_document_structure()
    industry = data.get("industry", "other")
    rules = INDUSTRY_RULES.get(industry, INDUSTRY_RULES["other"])

    brand_name = data.get("company_name", "Startup")
    audience = data.get("target_audience", "")
    mission = data.get("mission", "")
    vision = data.get("vision", "")
    values = data.get("core_values", [])
    personality = data.get("brand_personality", [])
    tone = data.get("tone_of_voice", "professional")
    tagline = data.get("tagline", "")

    audience_text = audience.rstrip(".")
    structure["Executive Summary"] = (
        f"{brand_name} is a {industry} startup serving {audience_text}. "
        f"The brand should communicate {rules['voice']} qualities with a consistent identity across digital and print touchpoints."
    )
    structure["Brand Strategy"] = {
        "purpose": f"Why the brand exists: {mission or 'Define the startup purpose clearly.'}",
        "vision": f"Desired future state: {vision or 'Describe the long-term impact the startup wants to create.'}",
        "mission": mission or "Insert the short operational mission here.",
        "core_values": values,
    }
    structure["Brand Identity"] = {
        "logo_guidelines": (
            f"Use the logo consistently on light and dark backgrounds. Maintain clear space, avoid distortion, "
            f"and keep a monochrome version available for restricted print use."
        ),
        "color_palette": recommend_color_palette(industry, data.get("color_preferences", "")),
        "typography": recommend_typography(industry, data.get("typography_preferences", "")),
        "imagery_style": data.get("imagery_style") or rules["imagery"],
    }
    structure["Brand Communication"] = {
        "voice": rules["voice"],
        "tone": tone,
        "tagline": tagline or f"Create a short slogan that reflects {brand_name}'s promise.",
    }
    structure["Brand Application"] = {
        "website": "Use the brand palette, type hierarchy, and logo clear space rules on the homepage and product pages.",
        "social_media": "Keep captions, cover images, and story assets consistent with the selected tone and color system.",
        "marketing_materials": "Apply the same visual identity to brochures, pitch decks, ads, and email headers.",
        "presentation": "Use branded title slides, section dividers, and icon styling.",
    }
    structure["Target Audience"] = audience
    structure["Brand Personality"] = personality
    structure["Visual Identity Guidelines"] = (
        f"Imagery should be {structure['Brand Identity']['imagery_style']}. "
        f"The visual system should feel {', '.join(personality) if personality else 'consistent and credible'} and support the {industry} context."
    )
    structure["Tone of Voice"] = (
        f"The brand should sound {tone}. Keep copy aligned with the audience and the startup stage."
    )
    structure["Brand Positioning"] = rules["positioning"]
    structure["Key Messaging"] = [
        f"What we do: {mission or 'Define the core offer.'}",
        f"Who we serve: {audience or 'Describe the target audience.'}",
        f"Why we matter: {vision or 'State the future impact.'}",
    ]
    structure["Usage Guidelines"] = [
        "Always use the approved logo variants.",
        "Use the approved palette across digital and print.",
        "Keep spacing, typography, and tone consistent.",
        "Adapt tone slightly by channel, but keep the same brand personality.",
    ]

    return structure


In [27]:
def render_brand_document(document: Dict[str, Any]) -> str:
    lines: List[str] = []
    lines.append("# Brand Guidelines")
    lines.append("")
    lines.append("## Executive Summary")
    lines.append(document.get("Executive Summary", "TBD"))
    lines.append("")
    lines.append("## Brand Strategy")
    lines.append(f"- **Purpose:** {document['Brand Strategy']['purpose']}")
    lines.append(f"- **Vision:** {document['Brand Strategy']['vision']}")
    lines.append(f"- **Mission:** {document['Brand Strategy']['mission']}")
    lines.append(f"- **Core Values:** {', '.join(document['Brand Strategy']['core_values']) or 'TBD'}")
    lines.append("")
    lines.append("## Brand Identity")
    lines.append(f"- **Logo Guidelines:** {document['Brand Identity']['logo_guidelines']}")
    lines.append(f"- **Color Palette:** {', '.join(document['Brand Identity']['color_palette'])}")
    lines.append(f"- **Typography:** {', '.join(document['Brand Identity']['typography'])}")
    lines.append(f"- **Imagery Style:** {document['Brand Identity']['imagery_style']}")
    lines.append("")
    logo_concepts = document.get("Brand Identity", {}).get("generated_logo_concepts", [])
    if logo_concepts:
        lines.append("## Logo Suggestions")
        lines.append("These logo concepts are generated starting points for manual refinement.")
        for idx, path in enumerate(logo_concepts, start=1):
            lines.append(f"- Concept {idx}: {path}")
        note = document.get("Brand Identity", {}).get("logo_generation_note")
        if note:
            lines.append(f"- Note: {note}")
        lines.append("")
    lines.append("## Brand Communication")
    lines.append(f"- **Voice:** {document['Brand Communication']['voice']}")
    lines.append(f"- **Tone:** {document['Brand Communication']['tone']}")
    lines.append(f"- **Tagline:** {document['Brand Communication']['tagline']}")
    lines.append("")
    lines.append("## Brand Application")
    for key, value in document["Brand Application"].items():
        lines.append(f"- **{key.replace('_', ' ').title()}:** {value}")
    lines.append("")
    lines.append("## Target Audience")
    lines.append(document.get("Target Audience", "TBD"))
    lines.append("")
    lines.append("## Brand Personality")
    lines.append(", ".join(document.get("Brand Personality", [])) or "TBD")
    lines.append("")
    lines.append("## Visual Identity Guidelines")
    lines.append(document.get("Visual Identity Guidelines", "TBD"))
    lines.append("")
    lines.append("## Tone of Voice")
    lines.append(document.get("Tone of Voice", "TBD"))
    lines.append("")
    lines.append("## Brand Positioning")
    lines.append(document.get("Brand Positioning", "TBD"))
    lines.append("")
    lines.append("## Key Messaging")
    for item in document.get("Key Messaging", []):
        lines.append(f"- {item}")
    lines.append("")
    lines.append("## Usage Guidelines")
    for item in document.get("Usage Guidelines", []):
        lines.append(f"- {item}")
    return "\n".join(lines)


sample_startup_profile = {
    "company_name": "Northstar Health",
    "industry": "health",
    "mission": "Make preventive care accessible through simple, human-centered digital services.",
    "vision": "Build the most trusted preventive health brand for busy urban professionals.",
    "target_audience": "Busy professionals aged 25-45 who want reliable, easy-to-use health services.",
    "competitors": ["Local clinics", "telehealth apps"],
    "tagline": "Care that fits your life.",
    "color_preferences": "calm green and soft blue",
    "typography_preferences": "clean modern sans serif",
    "imagery_style": "clean, bright, and human-centered",
    "brand_personality": ["trustworthy", "professional", "reassuring"],
    "tone_of_voice": "professional",
    "core_values": ["trust", "quality", "simplicity"],
    "usage_channels": ["website", "instagram", "linkedin", "presentations"],
}

brand_document = generate_brand_document(sample_startup_profile)
brand_document_markdown = render_brand_document(brand_document)

print(json.dumps(brand_document, indent=2, ensure_ascii=False))
print("\n" + "=" * 80 + "\n")
print(brand_document_markdown)


{
  "Executive Summary": "Northstar Health is a health startup serving Busy professionals aged 25-45 who want reliable, easy-to-use health services.. The brand should communicate reassuring, expert, calm qualities with a consistent identity across digital and print touchpoints.",
  "Brand Strategy": {
    "purpose": "Why the brand exists: Make preventive care accessible through simple, human-centered digital services.",
    "vision": "Desired future state: Build the most trusted preventive health brand for busy urban professionals.",
    "mission": "Make preventive care accessible through simple, human-centered digital services.",
    "core_values": [
      "trust",
      "quality",
      "simplicity"
    ]
  },
  "Brand Identity": {
    "logo_guidelines": "Use the logo consistently on light and dark backgrounds. Maintain clear space, avoid distortion, and keep a monochrome version available for restricted print use.",
    "color_palette": [
      "#2563EB",
      "#16A34A",
      "#0F

In [37]:
def export_brand_document(document: Dict[str, Any], output_dir: str = "outputs", base_name: str = "brand_guidelines") -> Dict[str, str]:
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    timestamp = date.today().isoformat()
    metadata = {
        "created_at": timestamp,
        "version": "1.0",
        "generator": "rule_based_branding_document_generator",
    }

    document_with_metadata = {
        "metadata": metadata,
        "document": document,
    }

    json_path = output_path / f"{base_name}.json"
    md_path = output_path / f"{base_name}.md"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(document_with_metadata, f, indent=2, ensure_ascii=False)
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(render_brand_document(document))

    docx_path: Path | None = None
    pdf_path = output_path / f"{base_name}.pdf"

    try:
        from docx import Document

        docx_path = output_path / f"{base_name}.docx"
        doc = Document()
        doc.add_heading("Brand Guidelines", level=0)
        doc.add_paragraph(f"Created: {timestamp}")
        doc.add_paragraph(f"Version: {metadata['version']}")
        doc.add_paragraph("")
        for line in render_brand_document(document).splitlines():
            if line.startswith("# "):
                doc.add_heading(line[2:], level=1)
            elif line.startswith("## "):
                doc.add_heading(line[3:], level=2)
            elif line.startswith("- "):
                doc.add_paragraph(line[2:])
            elif line.strip():
                doc.add_paragraph(line)
        doc.save(docx_path)
    except Exception:
        docx_path = None

    def _extract_startup_name(text: str) -> str:
        text = normalize_text(text)
        if not text:
            return "Brand"
        first_sentence = text.split(".")[0]
        first_chunk = first_sentence.split(" is ")[0].strip()
        return first_chunk[:60] if first_chunk else "Brand"

    def _wrap_lines(text: str, width: int = 100) -> List[str]:
        import textwrap
        return textwrap.wrap(text, width=width) if text else [""]

    def _draw_section(ax, y: float, title: str, content_lines: List[str], accent: str = "#8B5A2B") -> float:
        ax.text(0.06, y, title, fontsize=13, fontweight="bold", color=accent, va="top")
        y -= 0.03
        for line in content_lines:
            if not line:
                y -= 0.008
                continue
            wrapped = _wrap_lines(line, width=92)
            for piece in wrapped:
                ax.text(0.075, y, piece, fontsize=8.8, color="#1f2937", va="top")
                y -= 0.016
        y -= 0.01
        return y

    def _format_logo_page(logo_paths: List[str]):
        import matplotlib.pyplot as plt
        from matplotlib.gridspec import GridSpec

        fig = plt.figure(figsize=(8.27, 11.69))
        fig.patch.set_facecolor("white")
        gs = GridSpec(3, 2, figure=fig, hspace=0.25, wspace=0.18)
        fig.text(0.06, 0.975, "Logo Suggestions", fontsize=18, fontweight="bold", color="#4b2f1a", va="top")
        fig.text(0.06, 0.952, "Generated starting points for manual refinement", fontsize=10, color="#6b7280", va="top")

        axes = []
        for i in range(6):
            axes.append(fig.add_subplot(gs[i]))
        for ax, path in zip(axes, logo_paths):
            ax.axis("off")
            try:
                img = Image.open(path).convert("RGB")
                ax.imshow(img)
                ax.set_title(Path(path).name, fontsize=8, color="#374151")
            except Exception:
                ax.text(0.5, 0.5, Path(path).name, ha="center", va="center", fontsize=8)
        for ax in axes[len(logo_paths):]:
            ax.axis("off")
        return fig

    try:
        import matplotlib.pyplot as plt
        from matplotlib.backends.backend_pdf import PdfPages

        pdf_text = render_brand_document(document)
        logo_concepts = document.get("Brand Identity", {}).get("generated_logo_concepts", [])
        startup_name = _extract_startup_name(document.get("Executive Summary", ""))
        brand_identity = document.get("Brand Identity", {})
        brand_strategy = document.get("Brand Strategy", {})
        brand_comm = document.get("Brand Communication", {})
        brand_app = document.get("Brand Application", {})

        with PdfPages(pdf_path) as pdf:
            # Cover page
            fig = plt.figure(figsize=(8.27, 11.69))
            fig.patch.set_facecolor("#F7F3EE")
            ax = fig.add_axes([0, 0, 1, 1])
            ax.axis("off")
            ax.add_patch(plt.Rectangle((0.06, 0.83), 0.88, 0.10, color="#8B5A2B", alpha=0.12, transform=ax.transAxes, ec="none"))
            ax.text(0.08, 0.93, startup_name, fontsize=26, fontweight="bold", color="#4b2f1a", va="top")
            ax.text(0.08, 0.885, "Brand Guidelines", fontsize=18, color="#1f2937", va="top")
            ax.text(0.08, 0.855, f"Created: {timestamp}", fontsize=10, color="#6b7280", va="top")
            ax.text(0.08, 0.80, "Document Overview", fontsize=13, fontweight="bold", color="#8B5A2B", va="top")
            overview = [
                brand_strategy.get("purpose", ""),
                brand_identity.get("imagery_style", ""),
                brand_comm.get("tone", ""),
                brand_app.get("website", ""),
            ]
            y = 0.77
            for item in overview:
                for piece in _wrap_lines(item, width=90):
                    ax.text(0.09, y, f"• {piece}", fontsize=9.5, color="#1f2937", va="top")
                    y -= 0.024
            ax.text(0.08, 0.11, "Logo suggestions and detailed sections follow in the next pages.", fontsize=10, color="#6b7280", va="top")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

            # Main narrative pages
            section_groups = [
                (
                    "Brand Strategy",
                    [
                        f"Purpose: {brand_strategy.get('purpose', 'TBD')}",
                        f"Vision: {brand_strategy.get('vision', 'TBD')}",
                        f"Mission: {brand_strategy.get('mission', 'TBD')}",
                        f"Core Values: {', '.join(brand_strategy.get('core_values', [])) or 'TBD'}",
                    ],
                ),
                (
                    "Brand Identity",
                    [
                        f"Logo Guidelines: {brand_identity.get('logo_guidelines', 'TBD')}",
                        f"Color Palette: {', '.join(brand_identity.get('color_palette', [])) or 'TBD'}",
                        f"Typography: {', '.join(brand_identity.get('typography', [])) or 'TBD'}",
                        f"Imagery Style: {brand_identity.get('imagery_style', 'TBD')}",
                    ],
                ),
                (
                    "Brand Communication",
                    [
                        f"Voice: {brand_comm.get('voice', 'TBD')}",
                        f"Tone: {brand_comm.get('tone', 'TBD')}",
                        f"Tagline: {brand_comm.get('tagline', 'TBD')}",
                    ],
                ),
                (
                    "Brand Application",
                    [
                        f"Website: {brand_app.get('website', 'TBD')}",
                        f"Social Media: {brand_app.get('social_media', 'TBD')}",
                        f"Marketing Materials: {brand_app.get('marketing_materials', 'TBD')}",
                        f"Presentation: {brand_app.get('presentation', 'TBD')}",
                    ],
                ),
            ]

            fig = plt.figure(figsize=(8.27, 11.69))
            fig.patch.set_facecolor("white")
            ax = fig.add_axes([0, 0, 1, 1])
            ax.axis("off")
            ax.text(0.06, 0.965, "Brand Guidelines", fontsize=18, fontweight="bold", color="#4b2f1a", va="top")
            ax.text(0.06, 0.942, f"{startup_name} | {timestamp}", fontsize=9, color="#6b7280", va="top")
            y = 0.90
            for title, content_lines in section_groups:
                y = _draw_section(ax, y, title, content_lines)
                if y < 0.12:
                    pdf.savefig(fig, bbox_inches="tight")
                    plt.close(fig)
                    fig = plt.figure(figsize=(8.27, 11.69))
                    fig.patch.set_facecolor("white")
                    ax = fig.add_axes([0, 0, 1, 1])
                    ax.axis("off")
                    ax.text(0.06, 0.965, "Brand Guidelines", fontsize=18, fontweight="bold", color="#4b2f1a", va="top")
                    ax.text(0.06, 0.942, f"{startup_name} | {timestamp}", fontsize=9, color="#6b7280", va="top")
                    y = 0.90
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

            # Logo suggestion page
            if logo_concepts:
                fig = _format_logo_page(logo_concepts)
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)
    except Exception:
        pdf_path = None

    return {
        "json": str(json_path),
        "markdown": str(md_path),
        "docx": str(docx_path) if docx_path else "",
        "pdf": str(pdf_path) if pdf_path and pdf_path.exists() else "",
    }


exported_files = export_brand_document(brand_document)
print("Exported files:")
print(json.dumps(exported_files, indent=2))

Exported files:
{
  "json": "outputs\\brand_guidelines.json",
  "markdown": "outputs\\brand_guidelines.md",
  "docx": "",
  "pdf": "outputs\\brand_guidelines.pdf"
}


## Next Steps and Limitations

This notebook is intentionally **rule-based** so it can generate a consistent first-pass branding document without an LLM.

### What is ready
- Startup input form
- Structured parsing and validation
- Branding document template
- Rule-based content generator
- Export to JSON and Markdown, with optional DOCX/PDF support if the libraries are installed

### What remains optional
- Connect a Kaggle dataset for logo and visual references
- Add startup-specific brand style clustering
- Add manual review and approval flow before export

### Useful Kaggle dataset for this module
- **Recommended**: `siddharthkumarsah/logo-dataset-2341-classes-and-167140-images`
- **Why**: it is the most relevant dataset for logo and identity references
- **Optional**: social media datasets can help with application examples, but they are not required for the branding document itself


## Pipeline Architecture

The branding document generator is a deterministic pipeline that turns a short startup brief into a complete brand charter and export bundle.

```mermaid
flowchart TD
    A[Startup brief] --> B[Parse brief and optional hints]
    B --> C[Infer industry, audience, tone, and brand direction]
    C --> D[Apply rule-based brand templates]
    D --> E[Build brand strategy, identity, communication, and usage sections]
    E --> F[Optionally load Kaggle logo dataset]
    F --> G[Train logo generator and extract palette]
    G --> H[Generate multi-style logo concepts]
    H --> I[Attach logo suggestions to document]
    I --> J[Render Markdown / PDF / optional DOCX]
```

### Full Flow
- Input: short startup name plus a short description of what the startup does.
- Optional hints: color preference, tagline/message, and an industry override.
- Inference: the notebook infers industry, audience, mission, vision, positioning, and visual identity rules.
- Generation: it fills every branding section with structured copy based on the inferred profile.
- Visual layer: it can use the Kaggle logo dataset to create logo suggestions with multiple design families.
- Export: the final brand charter is exported to Markdown, JSON, and PDF.

### Module Breakdown
- **Input parsing**: accepts a brief, not a full brand worksheet.
- **Brand engine**: expands the brief into a complete document.
- **Logo module**: creates style-varied logo concepts from learned palette and layout templates.
- **Renderer**: formats the output for human review.
- **Exporter**: writes the final files into the `outputs` folder.


## Logo Generation for Brand Charter (Kaggle)

This section trains a lightweight logo generator from the Kaggle logo dataset and injects generated logo concept assets into the brand charter.

Pipeline:
1. Download/load dataset from Kaggle (`siddharthkumarsah/logo-dataset-2341-classes-and-167140-images`)
2. Build image subset for training
3. Train a compact VAE generator (CPU-friendly baseline)
4. Generate candidate logo concepts
5. Add generated assets to the branding document output


In [17]:
import random
import time
import subprocess
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps, ImageFilter
from torch.utils.data import Dataset, DataLoader

LOGO_DATASET_SLUG = "siddharthkumarsah/logo-dataset-2341-classes-and-167140-images"
ASSET_DIR = Path("outputs") / "branding_assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 128
OUTPUT_IMAGE_SIZE = 512
MAX_LOGO_IMAGES = 3000
BATCH_SIZE = 16
LATENT_DIM = 128
EPOCHS = 8
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

KAGGLE_RETRIES = 5
KAGGLE_WAIT_SECONDS = 15


def _is_image_file(path: Path) -> bool:
    return path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".bmp"}


def _collect_logo_images(root: Path, limit: int = MAX_LOGO_IMAGES) -> List[Path]:
    image_paths: List[Path] = []
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.bmp"):
        image_paths.extend(root.rglob(ext))
    image_paths = [p for p in image_paths if p.is_file()]
    random.shuffle(image_paths)
    return image_paths[:limit]


def _looks_like_dataset_root(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    sample = _collect_logo_images(path, limit=20)
    return len(sample) > 0


def _download_with_kagglehub(slug: str, retries: int = KAGGLE_RETRIES) -> Path | None:
    try:
        import kagglehub
    except Exception:
        return None

    for attempt in range(1, retries + 1):
        try:
            print(f"[kagglehub] Attempt {attempt}/{retries}...")
            downloaded = Path(kagglehub.dataset_download(slug))
            if _looks_like_dataset_root(downloaded):
                return downloaded
        except Exception as exc:
            print(f"[kagglehub] Attempt {attempt} failed: {exc}")
            if attempt < retries:
                sleep_s = KAGGLE_WAIT_SECONDS * attempt
                print(f"Waiting {sleep_s}s before retry...")
                time.sleep(sleep_s)
    return None


def _download_with_kaggle_cli(slug: str) -> Path | None:
    target_dir = Path("datasets") / "logo-dataset-2341-classes-and-167140-images"
    target_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "kaggle",
        "datasets",
        "download",
        "-d",
        slug,
        "-p",
        str(target_dir),
        "--unzip",
        "--force",
    ]
    try:
        print("[kaggle-cli] Trying Kaggle CLI fallback download...")
        subprocess.run(cmd, check=True, capture_output=True, text=True)
        return target_dir if _looks_like_dataset_root(target_dir) else None
    except Exception as exc:
        print(f"[kaggle-cli] Fallback failed: {exc}")
        return None


def _find_cached_dataset_roots() -> List[Path]:
    candidates: List[Path] = []

    local_fallback = Path("datasets") / "logo-dataset-2341-classes-and-167140-images"
    if local_fallback.exists():
        candidates.append(local_fallback)

    cache_base = Path.home() / ".cache" / "kagglehub" / "datasets" / "siddharthkumarsah" / "logo-dataset-2341-classes-and-167140-images"
    if cache_base.exists():
        candidates.append(cache_base)
        for sub in cache_base.rglob("*"):
            if sub.is_dir():
                candidates.append(sub)

    seen = set()
    uniq: List[Path] = []
    for c in candidates:
        k = str(c.resolve()) if c.exists() else str(c)
        if k not in seen:
            seen.add(k)
            uniq.append(c)
    return uniq


def resolve_logo_dataset_path() -> Path | None:
    path = _download_with_kagglehub(LOGO_DATASET_SLUG, retries=KAGGLE_RETRIES)
    if path is not None and _looks_like_dataset_root(path):
        return path

    path = _download_with_kaggle_cli(LOGO_DATASET_SLUG)
    if path is not None and _looks_like_dataset_root(path):
        return path

    for candidate in _find_cached_dataset_roots():
        if _looks_like_dataset_root(candidate):
            print(f"Using cached dataset folder: {candidate}")
            return candidate

    return None


logo_dataset_path = resolve_logo_dataset_path()

if logo_dataset_path is None:
    print("Logo dataset path not found after retries.")
    print("You can run later after network stabilizes, or place the dataset manually under datasets/logo-dataset-2341-classes-and-167140-images")
    logo_image_paths = []
else:
    logo_image_paths = _collect_logo_images(logo_dataset_path, limit=MAX_LOGO_IMAGES)
    print(f"Logo dataset root: {logo_dataset_path}")
    print(f"Collected {len(logo_image_paths)} images for training subset")

[kagglehub] Attempt 1/5...
Logo dataset root: C:\Users\MSI\.cache\kagglehub\datasets\siddharthkumarsah\logo-dataset-2341-classes-and-167140-images\versions\1
Collected 3000 images for training subset


In [19]:
class LogoDataset(Dataset):
    def __init__(self, image_paths: List[Path], size: int = IMAGE_SIZE):
        self.image_paths = image_paths
        self.size = size

    def __len__(self):
        return len(self.image_paths)

    def _load_image(self, path: Path) -> torch.Tensor:
        img = Image.open(path).convert("RGB")
        img = ImageOps.contain(img, (self.size, self.size))
        canvas = Image.new("RGB", (self.size, self.size), (255, 255, 255))
        x = (self.size - img.width) // 2
        y = (self.size - img.height) // 2
        canvas.paste(img, (x, y))
        arr = np.asarray(canvas).astype(np.float32) / 255.0
        arr = (arr * 2.0) - 1.0
        return torch.from_numpy(arr).permute(2, 0, 1)

    def __getitem__(self, idx: int):
        path = self.image_paths[idx]
        return self._load_image(path)


class ConvVAE(nn.Module):
    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1), nn.ReLU(inplace=True),
        )
        self.enc_out_dim = 512 * 4 * 4
        self.fc_mu = nn.Linear(self.enc_out_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.enc_out_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, self.enc_out_dim)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Tanh(),
        )

    def encode(self, x):
        h = self.encoder(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z).view(-1, 512, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


def vae_loss(recon_x, x, mu, logvar):
    mse = F.mse_loss(recon_x, x, reduction="mean")
    l1 = F.l1_loss(recon_x, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return (0.7 * mse) + (0.3 * l1) + (0.0008 * kl), mse.detach().item(), kl.detach().item()


print("Dataset and upgraded VAE model classes are ready.")

Dataset and upgraded VAE model classes are ready.


In [43]:
import cv2
from PIL import ImageDraw, ImageFont


STYLE_PRESETS = [
    "minimal_emblem",
    "lettermark_square",
    "orbital_mark",
    "shield_mark",
    "wordmark_with_symbol",
    "monoline_geometry",
]


def _auto_crop_foreground(img: Image.Image, near_white_thresh: int = 245) -> Image.Image:
    arr = np.asarray(img.convert("RGB"))
    fg_mask = np.any(arr < near_white_thresh, axis=2)
    ys, xs = np.where(fg_mask)
    if len(xs) == 0 or len(ys) == 0:
        return img
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    return img.crop((x0, y0, x1 + 1, y1 + 1))


def _enhance_logo_image(img: Image.Image, out_size: int = OUTPUT_IMAGE_SIZE) -> Image.Image:
    cropped = _auto_crop_foreground(img)
    padded = ImageOps.pad(cropped, (out_size, out_size), color=(255, 255, 255), centering=(0.5, 0.5))
    return padded.filter(ImageFilter.UnsharpMask(radius=2, percent=180, threshold=2))


def _logo_quality_score(img: Image.Image) -> float:
    arr = np.asarray(img.convert("RGB"))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    contrast = float(gray.std())
    non_white = float(np.mean(np.any(arr < 245, axis=2)))
    return (0.55 * sharpness) + (0.25 * contrast) + (20.0 * non_white)


def save_tensor_image(tensor: torch.Tensor, out_path: Path) -> None:
    x = tensor.detach().cpu().permute(1, 2, 0).numpy()
    x = np.clip((x + 1.0) / 2.0, 0.0, 1.0)
    img = (x * 255).astype(np.uint8)
    Image.fromarray(img).save(out_path)


def train_logo_generator(image_paths: List[Path], epochs: int = EPOCHS) -> Dict[str, Any]:
    if len(image_paths) < 50:
        raise RuntimeError("Not enough logo images for training. Need at least 50 images.")

    ds = LogoDataset(image_paths)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    model = ConvVAE(latent_dim=LATENT_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history = []
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch in dl:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            recon, mu, logvar = model(batch)
            loss, recon_loss, kl_loss = vae_loss(recon, batch, mu, logvar)
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.item())

        mean_loss = epoch_loss / max(1, len(dl))
        history.append({"epoch": epoch + 1, "loss": mean_loss})
        print(f"Epoch {epoch + 1}/{epochs} - loss: {mean_loss:.6f}")

    model_path = ASSET_DIR / "logo_vae.pt"
    torch.save(model.state_dict(), model_path)

    return {
        "model": model,
        "model_path": str(model_path),
        "history": history,
    }


def _extract_dominant_palette(image_paths: List[Path], max_images: int = 120, colors: int = 5) -> List[tuple[int, int, int]]:
    sampled = image_paths[:max_images]
    pixels = []
    for p in sampled:
        try:
            img = Image.open(p).convert("RGB").resize((64, 64))
            arr = np.asarray(img)
            arr = arr.reshape(-1, 3)
            arr = arr[np.any(arr < 245, axis=1)]
            if len(arr) > 0:
                pixels.append(arr)
        except Exception:
            continue

    if not pixels:
        return [(17, 24, 39), (37, 99, 235), (248, 250, 252), (15, 118, 110), (219, 39, 119)]

    all_pixels = np.concatenate(pixels, axis=0).astype(np.float32)
    if len(all_pixels) > 50000:
        idx = np.random.choice(len(all_pixels), 50000, replace=False)
        all_pixels = all_pixels[idx]

    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 0.2)
    _compactness, labels, centers = cv2.kmeans(all_pixels, colors, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
    centers = centers.astype(np.uint8)

    counts = np.bincount(labels.flatten(), minlength=colors)
    order = np.argsort(-counts)
    palette = [tuple(int(v) for v in centers[i]) for i in order]
    return palette


def _pick_font(size: int) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    candidates = [
        r"C:\Windows\Fonts\segoeuib.ttf",
        r"C:\Windows\Fonts\arialbd.ttf",
        r"C:\Windows\Fonts\bahnschrift.ttf",
    ]
    for fp in candidates:
        try:
            return ImageFont.truetype(fp, size=size)
        except Exception:
            continue
    return ImageFont.load_default()


def _startup_initials(name: str) -> str:
    parts = [p for p in str(name).strip().split() if p]
    if not parts:
        return "BR"
    if len(parts) == 1:
        return parts[0][:2].upper()
    return (parts[0][0] + parts[1][0]).upper()


def _darken(color: tuple[int, int, int], factor: float = 0.72) -> tuple[int, int, int]:
    return tuple(max(0, min(255, int(c * factor))) for c in color)


def _safe_palette(palette: List[tuple[int, int, int]]) -> List[tuple[int, int, int]]:
    safe = []
    for color in palette:
        lum = (0.2126 * color[0]) + (0.7152 * color[1]) + (0.0722 * color[2])
        if lum > 170:
            safe.append(_darken(color, 0.58))
        else:
            safe.append(color)
    return safe


def _draw_logo_template(canvas_size: int, style_name: str, startup_name: str, initials: str, c1, c2, c3) -> Image.Image:
    img = Image.new("RGB", (canvas_size, canvas_size), (255, 255, 255))
    d = ImageDraw.Draw(img)
    w = canvas_size
    h = canvas_size
    cx, cy = w // 2, h // 2

    if style_name == "minimal_emblem":
        d.rounded_rectangle([130, 130, w - 130, h - 130], radius=170, outline=c1, width=24)
        d.ellipse([220, 220, w - 220, h - 220], outline=c2, width=16)
        font = _pick_font(size=290)
        box = d.textbbox((0, 0), initials, font=font)
        d.text(((w - (box[2] - box[0])) // 2, (h - (box[3] - box[1])) // 2 - 10), initials, fill=c1, font=font)

    elif style_name == "lettermark_square":
        d.rounded_rectangle([150, 150, w - 150, h - 150], radius=70, outline=c1, width=20)
        d.rectangle([220, 220, w - 220, h - 220], outline=c2, width=14)
        font = _pick_font(size=300)
        box = d.textbbox((0, 0), initials, font=font)
        d.text(((w - (box[2] - box[0])) // 2, (h - (box[3] - box[1])) // 2 - 6), initials, fill=c3, font=font)

    elif style_name == "orbital_mark":
        d.ellipse([140, 140, w - 140, h - 140], outline=c1, width=22)
        d.arc([190, 220, w - 190, h - 120], start=15, end=338, fill=c2, width=20)
        d.arc([220, 160, w - 220, h - 190], start=210, end=30, fill=c3, width=18)
        font = _pick_font(size=270)
        box = d.textbbox((0, 0), initials, font=font)
        d.text(((w - (box[2] - box[0])) // 2, (h - (box[3] - box[1])) // 2 - 12), initials, fill=c1, font=font)

    elif style_name == "shield_mark":
        shield = [(cx, 120), (w - 170, 200), (w - 220, h - 240), (cx, h - 110), (220, h - 240), (170, 200)]
        d.polygon(shield, outline=c1, fill=(255, 255, 255), width=20)
        font = _pick_font(size=250)
        box = d.textbbox((0, 0), initials, font=font)
        d.text(((w - (box[2] - box[0])) // 2, (h - (box[3] - box[1])) // 2 + 10), initials, fill=c2, font=font)
        d.line([(260, h - 220), (w - 260, h - 220)], fill=c3, width=12)

    elif style_name == "wordmark_with_symbol":
        d.rounded_rectangle([120, 200, 360, 440], radius=55, outline=c1, width=20)
        icon_font = _pick_font(size=140)
        ib = d.textbbox((0, 0), initials[:1], font=icon_font)
        d.text((240 - (ib[2] - ib[0]) // 2, 320 - (ib[3] - ib[1]) // 2), initials[:1], fill=c2, font=icon_font)
        word_font = _pick_font(size=95)
        brand = startup_name.upper()[:16]
        wb = d.textbbox((0, 0), brand, font=word_font)
        d.text((430, 260), brand, fill=c1, font=word_font)
        d.line([(430, 390), (430 + min(420, wb[2] - wb[0]), 390)], fill=c3, width=10)

    else:
        d.line([(170, 170), (w - 170, 170)], fill=c1, width=18)
        d.line([(170, h - 170), (w - 170, h - 170)], fill=c1, width=18)
        d.line([(170, 170), (170, h - 170)], fill=c2, width=18)
        d.line([(w - 170, 170), (w - 170, h - 170)], fill=c2, width=18)
        d.arc([260, 260, w - 260, h - 260], start=0, end=360, fill=c3, width=18)
        font = _pick_font(size=240)
        box = d.textbbox((0, 0), initials, font=font)
        d.text(((w - (box[2] - box[0])) // 2, (h - (box[3] - box[1])) // 2), initials, fill=c1, font=font)

    if style_name != "wordmark_with_symbol":
        sub_font = _pick_font(size=62)
        brand = startup_name.upper()[:24]
        sb = d.textbbox((0, 0), brand, font=sub_font)
        d.text(((w - (sb[2] - sb[0])) // 2, h - 120), brand, fill=(38, 38, 38), font=sub_font)

    return img


def generate_crisp_logo_concepts(
    startup_name: str,
    palette: List[tuple[int, int, int]],
    n: int = 6,
    out_size: int = OUTPUT_IMAGE_SIZE,
) -> List[str]:
    initials = _startup_initials(startup_name)
    paths: List[str] = []

    while len(palette) < 5:
        palette.append((37, 99, 235))
    palette = _safe_palette(palette)

    supersample = 2
    canvas_size = out_size * supersample

    for i in range(n):
        style_name = STYLE_PRESETS[i % len(STYLE_PRESETS)]
        c1 = palette[i % len(palette)]
        c2 = palette[(i + 1) % len(palette)]
        c3 = palette[(i + 2) % len(palette)]

        high_res = _draw_logo_template(canvas_size, style_name, startup_name, initials, c1, c2, c3)
        img = high_res.resize((out_size, out_size), Image.Resampling.LANCZOS)
        img = _enhance_logo_image(img, out_size=out_size)

        out_path = ASSET_DIR / f"crisp_logo_concept_{i+1:02d}.png"
        img.save(out_path)
        paths.append(str(out_path))

    return paths


def generate_logo_concepts(model: ConvVAE, n: int = 6, seed: int = 42, candidate_multiplier: int = 8) -> List[str]:
    rng = torch.Generator(device=DEVICE)
    rng.manual_seed(seed)

    model.eval()
    concept_paths: List[str] = []
    candidates: List[tuple[float, Image.Image]] = []

    candidate_count = max(n * candidate_multiplier, n)
    with torch.no_grad():
        z = torch.randn((candidate_count, LATENT_DIM), generator=rng, device=DEVICE)
        outputs = model.decode(z)

    for i in range(candidate_count):
        x = outputs[i].detach().cpu().permute(1, 2, 0).numpy()
        x = np.clip((x + 1.0) / 2.0, 0.0, 1.0)
        base_img = Image.fromarray((x * 255).astype(np.uint8))
        enhanced = _enhance_logo_image(base_img, out_size=OUTPUT_IMAGE_SIZE)
        score = _logo_quality_score(enhanced)
        candidates.append((score, enhanced))

    candidates.sort(key=lambda t: t[0], reverse=True)
    top = candidates[:n]

    for i, (_score, img) in enumerate(top, start=1):
        out_path = ASSET_DIR / f"generated_logo_concept_{i:02d}.png"
        img.save(out_path)
        concept_paths.append(str(out_path))

    return concept_paths


def add_logo_concepts_to_brand_doc(document: Dict[str, Any], concept_paths: List[str], mode: str) -> Dict[str, Any]:
    document = dict(document)
    identity = dict(document.get("Brand Identity", {}))
    identity["generated_logo_concepts"] = concept_paths
    identity["logo_generation_note"] = (
        f"Logo concepts generated in '{mode}' mode using the Kaggle logo dataset as training/reference source. "
        "Use these as design directions and finalize with manual designer refinement."
    )
    document["Brand Identity"] = identity

    usage = list(document.get("Usage Guidelines", []))
    usage.append("Review generated logo concepts with a designer before final lockup approval.")
    document["Usage Guidelines"] = usage
    return document


training_result = None
vae_logo_paths: List[str] = []
final_logo_paths: List[str] = []

if len(logo_image_paths) >= 50:
    training_result = train_logo_generator(logo_image_paths, epochs=EPOCHS)

    vae_logo_paths = generate_logo_concepts(training_result["model"], n=6, seed=42, candidate_multiplier=10)
    vae_quality = []
    for p in vae_logo_paths:
        try:
            vae_quality.append(_logo_quality_score(Image.open(p).convert("RGB")))
        except Exception:
            pass
    mean_vae_quality = float(np.mean(vae_quality)) if vae_quality else 0.0

    startup_name = sample_startup_profile.get("company_name", "Startup") if isinstance(sample_startup_profile, dict) else "Startup"
    palette = _extract_dominant_palette(logo_image_paths, max_images=150, colors=5)
    crisp_paths = generate_crisp_logo_concepts(startup_name=startup_name, palette=palette, n=6)

    if mean_vae_quality < 55.0:
        final_logo_paths = crisp_paths
        generation_mode = "crisp_charter_generator"
    else:
        final_logo_paths = vae_logo_paths
        generation_mode = "vae_generator"

    brand_document = add_logo_concepts_to_brand_doc(brand_document, final_logo_paths, mode=generation_mode)
    brand_document_markdown = render_brand_document(brand_document)
    exported_files = export_brand_document(brand_document, output_dir="outputs", base_name="brand_guidelines_with_logo")

    print(f"Selected mode: {generation_mode}")
    print(f"VAE mean quality score: {mean_vae_quality:.2f}")
    print("\nPrimary logo concept files:")
    for p in final_logo_paths:
        print("-", p)

    print("\n(Also saved VAE experimental outputs):")
    for p in vae_logo_paths:
        print("-", p)

    print("\nExported branding document with logo concepts:")
    print(json.dumps(exported_files, indent=2))
else:
    print("Skipped training: dataset not available or insufficient images.")


Epoch 1/8 - loss: 0.385211
Epoch 2/8 - loss: 0.273316
Epoch 3/8 - loss: 0.249648
Epoch 4/8 - loss: 0.240257
Epoch 5/8 - loss: 0.229516
Epoch 6/8 - loss: 0.223888
Epoch 7/8 - loss: 0.217640
Epoch 8/8 - loss: 0.203701
Selected mode: crisp_charter_generator
VAE mean quality score: 42.72

Primary logo concept files:
- outputs\branding_assets\crisp_logo_concept_01.png
- outputs\branding_assets\crisp_logo_concept_02.png
- outputs\branding_assets\crisp_logo_concept_03.png
- outputs\branding_assets\crisp_logo_concept_04.png
- outputs\branding_assets\crisp_logo_concept_05.png
- outputs\branding_assets\crisp_logo_concept_06.png

(Also saved VAE experimental outputs):
- outputs\branding_assets\generated_logo_concept_01.png
- outputs\branding_assets\generated_logo_concept_02.png
- outputs\branding_assets\generated_logo_concept_03.png
- outputs\branding_assets\generated_logo_concept_04.png
- outputs\branding_assets\generated_logo_concept_05.png
- outputs\branding_assets\generated_logo_concept_06.pn

In [44]:
# Final test case: ScaleUP
scaleup_profile = {
    "company_name": "ScaleUP",
    "industry": "technology",
    "mission": "Deliver practical AI projects and consulting that help companies scale operations and decision quality.",
    "vision": "Become the trusted AI consulting partner for growth-stage companies in North Africa and Europe.",
    "target_audience": "SMEs and growth-stage companies seeking AI implementation, automation, and data strategy support.",
    "competitors": ["AI consulting agencies", "freelance data science teams"],
    "tagline": "Scale smarter with AI.",
    "color_preferences": "beige, brown",
    "typography_preferences": "clean modern sans-serif for headings and readable sans-serif for body",
    "imagery_style": "professional, minimalist, consultancy-oriented",
    "brand_personality": ["professional", "trustworthy", "innovative"],
    "tone_of_voice": "professional",
    "core_values": ["quality", "trust", "innovation"],
    "usage_channels": ["website", "linkedin", "presentations", "ads"],
}

scaleup_document = generate_brand_document(scaleup_profile)
scaleup_markdown = render_brand_document(scaleup_document)

# Optional logo concept integration if helper exists
if "logo_image_paths" in globals() and isinstance(logo_image_paths, list) and len(logo_image_paths) >= 50:
    try:
        palette = _extract_dominant_palette(logo_image_paths, max_images=150, colors=5)
        # Force first two colors to match requested preference direction.
        palette = [(210, 180, 140), (139, 99, 61)] + palette
        scaleup_logo_paths = generate_crisp_logo_concepts(startup_name="ScaleUP", palette=palette[:5], n=6)
        scaleup_document = add_logo_concepts_to_brand_doc(scaleup_document, scaleup_logo_paths, mode="crisp_charter_generator")
        scaleup_markdown = render_brand_document(scaleup_document)
    except Exception as exc:
        print(f"Logo concept generation skipped for test case: {exc}")

scaleup_exported = export_brand_document(scaleup_document, output_dir="outputs", base_name="brand_guidelines_scaleup")

print("ScaleUP test export:")
print(json.dumps(scaleup_exported, indent=2))
print("\nPreview:\n")
print(scaleup_markdown[:2000])

ScaleUP test export:
{
  "json": "outputs\\brand_guidelines_scaleup.json",
  "markdown": "outputs\\brand_guidelines_scaleup.md",
  "docx": "",
  "pdf": "outputs\\brand_guidelines_scaleup.pdf"
}

Preview:

# Brand Guidelines

## Executive Summary
ScaleUP is a technology startup serving SMEs and growth-stage companies seeking AI implementation, automation, and data strategy support. The brand should communicate smart, clear, innovative qualities with a consistent identity across digital and print touchpoints.

## Brand Strategy
- **Purpose:** Why the brand exists: Deliver practical AI projects and consulting that help companies scale operations and decision quality.
- **Vision:** Desired future state: Become the trusted AI consulting partner for growth-stage companies in North Africa and Europe.
- **Mission:** Deliver practical AI projects and consulting that help companies scale operations and decision quality.
- **Core Values:** quality, trust, innovation

## Brand Identity
- **Logo Gu